# DermaScope — model verification

Checks the retrained model end to end:

1. Rebuild the test split exactly as the training notebook does (scikit-learn 1.7.2) and save it.
2. Evaluate `best_model.keras` on the full test set — should reproduce 78.0% accuracy / 0.936 macro AUC.
3. Evaluate the existing `skin_cancer_model.tflite` using the Streamlit app's exact preprocessing.
4. Export an unquantized float32 TFLite and evaluate it the same way, then again with the app's updated preprocessing.
5. Grad-CAM on six test images, softmax score vs. pre-softmax logit.
6. Occlusion test on 200 test images, center masked vs. corners masked.

In [ ]:
# The training split only reproduces on some scikit-learn versions, so pin it
# before anything imports sklearn.
!pip install -q "scikit-learn==1.7.2"

In [ ]:
import os
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model
from PIL import Image
from scipy.stats import wilcoxon

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.preprocessing import LabelBinarizer

assert sklearn.__version__ == '1.7.2', f"Expected scikit-learn 1.7.2, got {sklearn.__version__}"

print("scikit-learn:", sklearn.__version__)
print("TensorFlow:  ", tf.__version__)
print("Keras:       ", keras.__version__)
print("pandas:      ", pd.__version__, "| numpy:", np.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU:         ", gpus[0].name if gpus else "⚠ No GPU")

In [ ]:
# Same settings as the training notebook's Config
IMG_SIZE    = 224
RANDOM_SEED = 42
N_FOLDS     = 20
OUT_DIR     = "/kaggle/working"

CLASSES = {
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevi',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis',
    'bkl':   'Benign Keratosis',
    'df':    'Dermatofibroma',
    'vasc':  'Vascular Lesion'
}
NUM_CLASSES = len(CLASSES)

# Numbers logged by the training run, used as pass/fail targets below
LOGGED_TEST_COUNTS = [162, 999, 53, 41, 169, 30, 30]
LOGGED_ACCURACY    = 0.7803
LOGGED_MACRO_AUC   = 0.9364

# SHA-256 of the model files committed in the GitHub repo
REPO_SHA256 = {
    'best_model.keras':         '2bf6c946860b2f50d9168d7f17d3f43fedbe79fe3b717c32563e43d6b0b7d9b9',
    'skin_cancer_model.tflite': 'd0f35b44e4b18c342dd5e9e6ede943994b4f1522f21d03a760c8e66715273447',
    'skin_cancer_model_f32.tflite': '56b733b9a331a7d8e1208616a232830109393884d5f9744e63ee46e78762816b',
}

# Locate inputs by filename so this works whatever the mount layout is
all_files = sorted(os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs)

def find_input(filename):
    hits = [p for p in all_files if os.path.basename(p) == filename]
    assert hits, f"{filename} not found under /kaggle/input"
    if len(hits) > 1:
        print(f"  note: {len(hits)} copies of {filename}, using the first")
    return hits[0]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

METADATA_PATH = find_input('HAM10000_metadata.csv')
KERAS_PATH    = find_input('best_model.keras')
TFLITE_PATH   = find_input('skin_cancer_model.tflite')

for p in (METADATA_PATH, KERAS_PATH, TFLITE_PATH):
    print(p)
for p in (KERAS_PATH, TFLITE_PATH):
    name = os.path.basename(p)
    same = sha256(p) == REPO_SHA256[name]
    print(f"{name}: {'identical to' if same else '⚠ DIFFERENT from'} the file in the repo")

## 1 — Rebuild the test split

In [ ]:
df = pd.read_csv(METADATA_PATH)

# Build full image paths (the dataset ships the images in more than one
# folder; duplicates are the same files, so the first hit is fine)
all_image_paths = {}
for p in all_files:
    if p.endswith('.jpg'):
        all_image_paths.setdefault(os.path.splitext(os.path.basename(p))[0], p)

df['path'] = df['image_id'].map(all_image_paths)
print(f"Images found: {df['path'].notnull().sum()} / {len(df)}")
df = df.dropna(subset=['path'])

label_to_idx = {dx: i for i, dx in enumerate(CLASSES.keys())}
idx_to_label = {i: dx for dx, i in label_to_idx.items()}
class_names  = [CLASSES[idx_to_label[i]] for i in range(NUM_CLASSES)]
df['label_idx'] = df['dx'].map(label_to_idx)

# Identical to the training notebook: 20 lesion-grouped stratified folds,
# 14 → train, 3 → val, 3 → test.
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_indices = [held_out_idx for _, held_out_idx in
                sgkf.split(df, df['label_idx'], groups=df['lesion_id'])]

train_idx = np.concatenate(fold_indices[0:14])
val_idx   = np.concatenate(fold_indices[14:17])
test_idx  = np.concatenate(fold_indices[17:20])

train_df = df.iloc[train_idx]
val_df   = df.iloc[val_idx]
test_df  = df.iloc[test_idx]
print(f"Train {len(train_df)} / Val {len(val_df)} / Test {len(test_df)}")

assert not set(test_df['lesion_id']) & (set(train_df['lesion_id']) | set(val_df['lesion_id'])), \
    "Lesion overlap between test and train/val!"

test_counts = test_df['label_idx'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).tolist()
print("Test per-class counts:", test_counts)
print("Logged by training:   ", LOGGED_TEST_COUNTS)
assert test_counts == LOGGED_TEST_COUNTS, "Test split does not match the training run — stop here."
print("✓ Test split matches the training run's per-class counts")

test_df.drop(columns=['path']).to_csv(f"{OUT_DIR}/test_split.csv", index=False)
print(f"Saved {OUT_DIR}/test_split.csv")

y_true     = test_df['label_idx'].values
y_true_bin = LabelBinarizer().fit_transform(y_true)
N_TEST     = len(test_df)

In [ ]:
results = []

def evaluate(name, probs, ref_pred=None):
    """Accuracy, macro AUC, per-class recall, and agreement with the Keras reference."""
    pred = np.argmax(probs, axis=1)
    row = {
        'model':     name,
        'accuracy':  accuracy_score(y_true, pred),
        'macro_auc': roc_auc_score(y_true_bin, probs, multi_class='ovr', average='macro'),
        'agreement_with_keras': np.nan if ref_pred is None else float(np.mean(pred == ref_pred)),
    }
    recalls = recall_score(y_true, pred, average=None, labels=list(range(NUM_CLASSES)))
    for i, r in enumerate(recalls):
        row[f"recall_{idx_to_label[i]}"] = r
    results.append(row)
    print(f"{name}: accuracy {row['accuracy']:.4f} | macro AUC {row['macro_auc']:.4f}"
          + ("" if ref_pred is None else f" | agreement {row['agreement_with_keras']:.4f}"))
    return pred

## 2 — `best_model.keras` on the full test set

Uses the training notebook's own test pipeline (TensorFlow JPEG decode + bilinear resize, raw 0–255 values).

In [ ]:
def load_test_dataset(dataframe, batch_size=32):
    """The training notebook's load_dataset() with augment=False, shuffle=False."""
    paths = dataframe['path'].values

    def load_and_preprocess(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        img = img * 255.0
        return img

    dataset = tf.data.Dataset.from_tensor_slices(paths)
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


best_model = keras.models.load_model(KERAS_PATH)

probs_keras = best_model.predict(load_test_dataset(test_df), verbose=1)
pred_keras  = evaluate("Keras (training preprocessing)", probs_keras)

acc_keras = results[-1]['accuracy']
auc_keras = results[-1]['macro_auc']
print(f"\nAccuracy  {acc_keras:.4f} vs logged {LOGGED_ACCURACY:.4f} → "
      f"{'✓ reproduced' if abs(acc_keras - LOGGED_ACCURACY) < 0.002 else '✗ NOT reproduced'}")
print(f"Macro AUC {auc_keras:.4f} vs logged {LOGGED_MACRO_AUC:.4f} → "
      f"{'✓ reproduced' if abs(auc_keras - LOGGED_MACRO_AUC) < 0.002 else '✗ NOT reproduced'}")

## 3 — Existing `skin_cancer_model.tflite` with the app's old preprocessing

`preprocess_image` and `run_inference` are copied programmatically from
`dermascope/skin_cancer_app.py` as of commit 8170423 (PIL Lanczos resize) — the
version the app used before its preprocessing was changed to match training.

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter   # what the app uses
    print("Interpreter: ai_edge_litert")
except ImportError:
    Interpreter = tf.lite.Interpreter
    print("Interpreter: tf.lite (ai_edge_litert not installed here)")


# ── Copied from dermascope/skin_cancer_app.py @ 8170423 (old preprocessing) ──
def preprocess_image(img: Image.Image) -> np.ndarray:
    """PIL Image → float32 array in [0, 255] ready for TFLite (model rescales internally)."""
    img = img.convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    return np.expand_dims(arr, axis=0)


def run_inference(interpreter, input_data: np.ndarray) -> np.ndarray:
    """Run TFLite inference. Returns probability array of shape (7,)."""
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    return output[0]
# ── End copied code ──


def preprocess_test_set():
    """Run every test image through the current preprocess_image(), the way the app does.
    The values are whole numbers 0–255, so they are cached as uint8 (checked to be
    lossless) and cast back to float32 at inference time."""
    X = np.empty((N_TEST, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for i, path in enumerate(test_df['path'].values):
        arr = preprocess_image(Image.open(path).convert("RGB"))
        X[i] = arr[0]
        assert np.array_equal(X[i].astype(np.float32), arr[0])
    return X


X_app = preprocess_test_set()
print("Old-app-preprocessed test set:", X_app.shape)


def tflite_predict(model_path, X):
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    print("  input dtype:", interpreter.get_input_details()[0]['dtype'].__name__,
          f"| file size: {os.path.getsize(model_path) / 1e6:.1f} MB")
    probs = np.empty((N_TEST, NUM_CLASSES), dtype=np.float32)
    for i in range(N_TEST):
        probs[i] = run_inference(interpreter, X[i:i + 1].astype(np.float32))
    return probs


# Extra row: the Keras model on the app's old preprocessing. This separates the
# effect of the app's resize (PIL Lanczos vs. TF bilinear) from the effect
# of the TFLite conversion.
probs_keras_app = best_model.predict(X_app.astype(np.float32), batch_size=32, verbose=1)
pred_keras_app  = evaluate("Keras (old app preprocessing)", probs_keras_app, ref_pred=pred_keras)

probs_tflite = tflite_predict(TFLITE_PATH, X_app)
pred_tflite  = evaluate("Existing TFLite (old app preprocessing)", probs_tflite, ref_pred=pred_keras)

## 4 — New float32 TFLite, no quantization

In [ ]:
# No converter.optimizations → weights stay float32 (the training notebook
# set tf.lite.Optimize.DEFAULT, which quantizes the weights).
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_f32 = converter.convert()

F32_PATH = f"{OUT_DIR}/skin_cancer_model_f32.tflite"
with open(F32_PATH, 'wb') as f:
    f.write(tflite_f32)
print(f"Saved {F32_PATH}")
print("skin_cancer_model_f32.tflite:",
      "identical to" if sha256(F32_PATH) == REPO_SHA256['skin_cancer_model_f32.tflite'] else "⚠ DIFFERENT from",
      "the file the app loads")

probs_f32 = tflite_predict(F32_PATH, X_app)
pred_f32  = evaluate("Float32 TFLite (old app preprocessing)", probs_f32, ref_pred=pred_keras)

### Float32 TFLite with the app's updated preprocessing

This is what the app runs now: the float32 model, with `preprocess_image` and
`run_inference` copied programmatically from the current `dermascope/skin_cancer_app.py`
(cv2 bilinear resize, matching the training notebook).

In [ ]:
# ── Copied from dermascope/skin_cancer_app.py (current, updated preprocessing) ──
def preprocess_image(img: Image.Image) -> np.ndarray:
    """PIL Image → float32 array in [0, 255], resized the way the training pipeline does."""
    arr = np.array(img.convert("RGB"), dtype=np.uint8)
    arr = cv2.resize(arr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    arr = arr.astype(np.float32)
    return np.expand_dims(arr, axis=0)


def run_inference(interpreter, input_data: np.ndarray) -> np.ndarray:
    """Run TFLite inference. Returns probability array of shape (7,)."""
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    return output[0]
# ── End copied code ──


X_app_new = preprocess_test_set()
print("Updated-app-preprocessed test set:", X_app_new.shape)

# How far is each app input from what the training pipeline feeds the model?
X_train_style = np.concatenate([b.numpy() for b in load_test_dataset(test_df.iloc[:64])])
print(f"Mean |pixel difference| vs training pipeline, first 64 images: "
      f"old app {np.abs(X_app[:64] - X_train_style).mean():.2f}, "
      f"updated app {np.abs(X_app_new[:64] - X_train_style).mean():.2f}  (0–255 scale)")

probs_f32_new = tflite_predict(F32_PATH, X_app_new)
pred_f32_new  = evaluate("Float32 TFLite (updated app preprocessing)", probs_f32_new, ref_pred=pred_keras)

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
results_df.to_csv(f"{OUT_DIR}/verification_metrics.csv")

print("=" * 60)
print("SUMMARY — full test set, n =", N_TEST)
print("=" * 60)
print(results_df[['accuracy', 'macro_auc', 'agreement_with_keras']].round(4).to_string())
print("\nPer-class recall:")
print(results_df[[c for c in results_df.columns if c.startswith('recall_')]].round(3).to_string())
print("\nAgreement between TFLite models and Keras on identical (old app) input:")
print(f"  Existing TFLite vs Keras: {np.mean(pred_tflite == pred_keras_app):.4f}")
print(f"  Float32 TFLite vs Keras:  {np.mean(pred_f32 == pred_keras_app):.4f}")

## 5 — Grad-CAM: softmax score vs. pre-softmax logit

Same six test images and the same `top_conv` layer as the training notebook. The only
difference between the two rows is which score is differentiated: the softmax
probability of the predicted class, or its logit before softmax.

In [ ]:
def load_cv2(path):
    """The training notebook's preprocess_image(): cv2 load → RGB → resize → float32 [0, 255]."""
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.astype(np.float32)


def overlay_gradcam(img_uint8, heatmap, alpha=0.4):
    """Overlay a Grad-CAM heatmap (0-1 float array) on an RGB uint8 image."""
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = img_uint8.astype(np.float32) * (1 - alpha) + heatmap_color.astype(np.float32) * alpha
    return np.uint8(np.clip(overlay, 0, 255))


# The final Dense layer applies softmax itself, so the logits are rebuilt
# from its input features and its own kernel/bias.
last_dense = best_model.layers[-1]
assert last_dense.activation.__name__ == 'softmax', "Expected the last layer to be Dense(softmax)"

grad_model = Model(
    best_model.inputs,
    [best_model.get_layer('top_conv').output, last_dense.input, best_model.output]
)


def gradcam_softmax_and_logit(img_batch):
    """Grad-CAM for the predicted class, once from the softmax score and once from the logit."""
    with tf.GradientTape(persistent=True) as tape:
        conv_outputs, features, predictions = grad_model(tf.convert_to_tensor(img_batch))
        logits = tf.matmul(features, last_dense.kernel) + last_dense.bias
        pred_index = int(tf.argmax(predictions[0]))
        scores = {'softmax': predictions[:, pred_index], 'logit': logits[:, pred_index]}

    out = {}
    for name, score in scores.items():
        grads = tape.gradient(score, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.squeeze(conv_outputs[0] @ pooled_grads[..., tf.newaxis])
        heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
        out[name] = (heatmap.numpy(), float(tf.reduce_mean(tf.abs(grads))))
    del tape

    softmax_check = float(tf.reduce_max(tf.abs(tf.nn.softmax(logits) - predictions)))
    return out, pred_index, float(predictions[0, pred_index]), softmax_check


# Same selection as the training notebook: one image per class, first six classes
sample_rows = pd.concat([test_df[test_df['label_idx'] == i].sample(1, random_state=RANDOM_SEED)
                         for i in range(6)])

fig, axes = plt.subplots(3, 6, figsize=(22, 12))
fig.suptitle('Grad-CAM (top_conv) — softmax score vs. pre-softmax logit', fontsize=14, fontweight='bold')

for col, (_, row) in enumerate(sample_rows.iterrows()):
    img = load_cv2(row['path'])
    cams, pred_idx, confidence, softmax_check = gradcam_softmax_and_logit(np.expand_dims(img, axis=0))
    assert softmax_check < 1e-4, "Rebuilt logits do not match the model output"

    cam_soft, grad_soft   = cams['softmax']
    cam_logit, grad_logit = cams['logit']
    corr = np.corrcoef(cam_soft.ravel(), cam_logit.ravel())[0, 1]
    print(f"{row['image_id']}  true {row['dx']:<5} pred {idx_to_label[pred_idx]:<5} conf {confidence:.3f} | "
          f"mean |grad| softmax {grad_soft:.2e}, logit {grad_logit:.2e} | heatmap correlation {corr:.3f}")

    axes[0][col].imshow(img.astype(np.uint8))
    axes[0][col].set_title(f"{CLASSES[row['dx']]}\n{row['image_id']}", fontsize=9, fontweight='bold')
    axes[1][col].imshow(overlay_gradcam(img.astype(np.uint8), cam_soft))
    axes[1][col].set_title(f"Softmax score\n(pred: {CLASSES[idx_to_label[pred_idx]]}, {confidence:.0%})", fontsize=8)
    axes[2][col].imshow(overlay_gradcam(img.astype(np.uint8), cam_logit))
    axes[2][col].set_title(f"Pre-softmax logit\n(heatmap corr. {corr:.2f})", fontsize=8)
    for r in range(3):
        axes[r][col].axis('off')

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/gradcam_softmax_vs_logit.png", dpi=150, bbox_inches='tight')
plt.show()

## 6 — Occlusion test: center masked vs. corners masked

200 test images. HAM10000 has no lesion masks, so the **image center is used as a stand-in
for the lesion** (lesions in this dataset are usually centered). Both masks cover the same
area — 25% of the image — and are filled with the image's mean color. If the model relies on
the lesion, masking the center should hurt far more than masking the corners.

In [ ]:
N_OCCLUSION = 200
occ_df = test_df.sample(N_OCCLUSION, random_state=RANDOM_SEED)
X_occ  = np.stack([load_cv2(p) for p in occ_df['path'].values])   # float32 [0, 255]

half, quarter = IMG_SIZE // 2, IMG_SIZE // 4
fill = X_occ.mean(axis=(1, 2), keepdims=True)                      # per-image mean color

center_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
center_mask[quarter:quarter + half, quarter:quarter + half] = True  # one 112x112 square

corner_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
corner_mask[:quarter, :quarter]   = True                            # four 56x56 squares
corner_mask[:quarter, -quarter:]  = True
corner_mask[-quarter:, :quarter]  = True
corner_mask[-quarter:, -quarter:] = True
assert center_mask.sum() == corner_mask.sum()

X_center = np.where(center_mask[None, :, :, None], fill, X_occ).astype(np.float32)
X_corner = np.where(corner_mask[None, :, :, None], fill, X_occ).astype(np.float32)

p_orig   = best_model.predict(X_occ,    batch_size=32, verbose=0)
p_center = best_model.predict(X_center, batch_size=32, verbose=0)
p_corner = best_model.predict(X_corner, batch_size=32, verbose=0)

top  = p_orig.argmax(axis=1)
rows = np.arange(N_OCCLUSION)
occ_results = pd.DataFrame({
    'image_id':      occ_df['image_id'].values,
    'dx':            occ_df['dx'].values,
    'pred_original': [idx_to_label[i] for i in top],
    'conf_original': p_orig[rows, top],
    'conf_center_masked': p_center[rows, top],
    'conf_corner_masked': p_corner[rows, top],
    'pred_center_masked': [idx_to_label[i] for i in p_center.argmax(axis=1)],
    'pred_corner_masked': [idx_to_label[i] for i in p_corner.argmax(axis=1)],
})
occ_results['drop_center'] = occ_results['conf_original'] - occ_results['conf_center_masked']
occ_results['drop_corner'] = occ_results['conf_original'] - occ_results['conf_corner_masked']
occ_results.to_csv(f"{OUT_DIR}/occlusion_results.csv", index=False)

flip_center = np.mean(p_center.argmax(axis=1) != top)
flip_corner = np.mean(p_corner.argmax(axis=1) != top)
stat = wilcoxon(occ_results['drop_center'], occ_results['drop_corner'])

print("=" * 60)
print(f"OCCLUSION TEST — n = {N_OCCLUSION}, 25% of image masked")
print("=" * 60)
print(f"{'':<28}{'center masked':>15}{'corners masked':>16}")
print(f"{'Mean confidence drop':<28}{occ_results['drop_center'].mean():>15.3f}{occ_results['drop_corner'].mean():>16.3f}")
print(f"{'Median confidence drop':<28}{occ_results['drop_center'].median():>15.3f}{occ_results['drop_corner'].median():>16.3f}")
print(f"{'Prediction changed':<28}{flip_center:>15.1%}{flip_corner:>16.1%}")
print(f"\nCenter mask hurt more than corner mask in "
      f"{np.mean(occ_results['drop_center'] > occ_results['drop_corner']):.1%} of images")
print(f"Wilcoxon signed-rank p-value (center drop vs corner drop): {stat.pvalue:.2e}")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Occlusion test — center vs. corners', fontsize=14, fontweight='bold')
axes[0].imshow(X_occ[0].astype(np.uint8));    axes[0].set_title('Original');       axes[0].axis('off')
axes[1].imshow(X_center[0].astype(np.uint8)); axes[1].set_title('Center masked');  axes[1].axis('off')
axes[2].imshow(X_corner[0].astype(np.uint8)); axes[2].set_title('Corners masked'); axes[2].axis('off')
axes[3].boxplot([occ_results['drop_center'], occ_results['drop_corner']])
axes[3].set_xticks([1, 2])
axes[3].set_xticklabels(['Center masked', 'Corners masked'])
axes[3].set_ylabel('Drop in confidence for the original prediction')
axes[3].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/occlusion_test.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("Files saved:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {OUT_DIR}/{f}")